In [18]:
import pandas as pd

# Load the CSV file
df = pd.read_csv("derm12345_metadata_all.csv")

# Display all unique options under the 'label' column
unique_labels = df['label'].unique()
unique_sub_classes = df['sub_class'].unique()
unique_main_class_2 = df['main_class_2'].unique()
unique_main_class_1 = df['main_class_1'].unique()
unique_malignancy = df['malignancy'].unique()
unique_super_class = df['super_class'].unique()


# Print them all
print(unique_labels)
print(unique_sub_classes)
print(unique_main_class_2)
print(unique_main_class_1)
print(unique_malignancy)
print(unique_super_class)

['acb' 'acd' 'ajb' 'ajd' 'ak' 'alm' 'angk' 'anm' 'bcc' 'bd' 'bdb' 'cb'
 'ccb' 'ccd' 'cd' 'ch' 'cjb' 'db' 'df' 'dfsp' 'ha' 'isl' 'jb' 'jd' 'ks'
 'la' 'lk' 'lm' 'lmm' 'ls' 'mcb' 'mel' 'mpd' 'pg' 'rd' 'sa' 'scc' 'sk'
 'sl' 'srjd']
['acral' 'actinic_keratosis' 'acral_lentiginious' 'angiokeratoma'
 'acral_nodular' 'basal_cell_carcinoma' 'bowen_disease' 'blue' 'compound'
 'congenital' 'cutaneous_horn' 'dermal' 'dermatofibroma'
 'dermatofibrosarcoma_protuberans' 'hemangioma' 'ink_spot_lentigo'
 'junctional' 'kaposi_sarcoma' 'lymphangioma' 'lichenoid_keratosis'
 'lentigo_maligna' 'lentigo_maligna_melanoma' 'lentigo_simplex'
 'miescher ' 'melanoma' 'mammary_paget_disease' 'pyogenic_granuloma'
 'recurrent' 'spider_angioma' 'squamous_cell_carcinoma'
 'seborrheic_keratosis' 'solar_lentigo' 'spitz_reed']
['compound' 'junctional' 'keratinocytic' 'melanoma' 'vascular' 'dermal'
 'fibro_histiocytic' 'lentigo' 'recurrent']
['banal' 'dysplastic' 'keratinocytic' 'melanoma' 'vascular'
 'fibro_histiocytic' 

In [13]:
# --- 0) Imports
import os, re, pathlib
import numpy as np

# --- 1) Load files
NPZ_PATH = "derm12345_google-derm-foundation-model_embeddings.npz"   # <- change if needed                             # <- change if needed

npz = np.load(NPZ_PATH, allow_pickle=True)
print("NPZ keys:", npz.files)


NPZ keys: ['derm12345_all_images/DERM_947970.jpg', 'derm12345_all_images/DERM_843871.jpg', 'derm12345_all_images/DERM_719083.jpg', 'derm12345_all_images/DERM_783499.jpg', 'derm12345_all_images/DERM_830572.jpg', 'derm12345_all_images/DERM_590657.jpg', 'derm12345_all_images/DERM_110590.jpg', 'derm12345_all_images/DERM_125494.jpg', 'derm12345_all_images/DERM_852220.jpg', 'derm12345_all_images/DERM_135726.jpg', 'derm12345_all_images/DERM_222838.jpg', 'derm12345_all_images/DERM_328318.jpg', 'derm12345_all_images/DERM_305870.jpg', 'derm12345_all_images/DERM_771195.jpg', 'derm12345_all_images/DERM_490428.jpg', 'derm12345_all_images/DERM_162768.jpg', 'derm12345_all_images/DERM_252202.jpg', 'derm12345_all_images/DERM_206677.jpg', 'derm12345_all_images/DERM_255498.jpg', 'derm12345_all_images/DERM_254008.jpg', 'derm12345_all_images/DERM_879187.jpg', 'derm12345_all_images/DERM_825774.jpg', 'derm12345_all_images/DERM_487693.jpg', 'derm12345_all_images/DERM_497899.jpg', 'derm12345_all_images/DERM_38

In [16]:
import pandas as pd

meta = pd.read_csv("derm12345_metadata_all.csv")

# 1) how many unique diagnoses per patient
diag_nunique = meta.groupby("patient_id")["label"].nunique()

# 2) patients with >1 diagnosis
conflict_patients = diag_nunique[diag_nunique > 1].sort_values(ascending=False)
print(f"Patients with multiple diagnoses: {len(conflict_patients)}")
display(conflict_patients.head(20))  # preview

# 3) show which diagnoses and counts for those patients
if len(conflict_patients) > 0:
    conflicted_df = (
        meta[meta["patient_id"].isin(conflict_patients.index)]
        .groupby(["patient_id", "label"])
        .size()
        .reset_index(name="count")
        .sort_values(["patient_id", "count"], ascending=[True, False])
    )
    display(conflicted_df.head(50))
else:
    print("No patient has multiple diagnoses 🎉")


Patients with multiple diagnoses: 782


patient_id
PID_146801    9
PID_960697    8
PID_512851    8
PID_533270    8
PID_442513    8
PID_897293    7
PID_464604    7
PID_268679    7
PID_130179    7
PID_542194    7
PID_299906    7
PID_478864    6
PID_380165    6
PID_662630    6
PID_154109    6
PID_466612    6
PID_143400    6
PID_160198    6
PID_586539    6
PID_382992    6
Name: label, dtype: int64

,patient_id,label,count
2,PID_100075,cjb,52
1,PID_100075,ccb,34
0,PID_100075,ajb,13
3,PID_100652,cb,2
4,PID_100652,jb,1
5,PID_100652,jd,1
6,PID_102277,cb,3
7,PID_102277,mcb,1
8,PID_103360,bdb,1
9,PID_103360,jb,1


In [19]:
import pandas as pd

# Load metadata
df = pd.read_csv("derm12345_metadata_all.csv")

# Normalize labels
df["label"] = df["label"].str.strip().str.lower()

# Target columns
CODES = ["NEV", "BCC", "ACK", "SEK", "SCC", "MEL"]
out = pd.DataFrame(df[["patient_id", "image_id"]].copy())
for c in CODES:
    out[c] = 0

# --- Mapping sets ---
# Benign nevi (Melanocytic benign)
nev_labels = {"cb", "bdb", "db", "jb", "cd", "jd"}  

# Cancers & keratoses
bcc_labels = {"bcc"}          # Basal Cell Carcinoma
ack_labels = {"ak"}           # Actinic Keratosis
sek_labels = {"sk"}           # Seborrheic Keratosis
scc_labels = {"scc"}          # Squamous Cell Carcinoma

# Melanoma and subtypes (always include lm)
mel_labels = {"mel", "alm", "anm", "lm", "lmm"}

# --- Assign one-hot flags ---
out.loc[df["label"].isin(nev_labels), "NEV"] = 1
out.loc[df["label"].isin(bcc_labels), "BCC"] = 1
out.loc[df["label"].isin(ack_labels), "ACK"] = 1
out.loc[df["label"].isin(sek_labels), "SEK"] = 1
out.loc[df["label"].isin(scc_labels), "SCC"] = 1
out.loc[df["label"].isin(mel_labels), "MEL"] = 1

# If MEL = 1, clear other columns to avoid overlap
out.loc[out["MEL"] == 1, ["NEV","BCC","ACK","SEK","SCC"]] = 0

# Keep only rows with any of the six targets
mask_any = (out[["NEV","BCC","ACK","SEK","SCC","MEL"]].sum(axis=1) > 0)
out = out[mask_any].reset_index(drop=True)

# Force integer 1/0
out[["NEV","BCC","ACK","SEK","SCC","MEL"]] = out[["NEV","BCC","ACK","SEK","SCC","MEL"]].astype(int)

# Save CSV
out.to_csv("derm12345_labels.csv", index=False)

# Quick summary
print("✅ derm12345_labels.csv created successfully!")
print(out[["NEV","BCC","ACK","SEK","SCC","MEL"]].sum().sort_values(ascending=False))
print(out.head())


✅ derm12345_labels.csv created successfully!
NEV    8441
SEK     607
BCC     423
MEL     400
SCC     266
ACK      58
dtype: int64
   patient_id     image_id  NEV  BCC  ACK  SEK  SCC  MEL
0  PID_124696  DERM_822769    0    0    1    0    0    0
1  PID_152688  DERM_172873    0    0    1    0    0    0
2  PID_152688  DERM_204720    0    0    1    0    0    0
3  PID_152688  DERM_355404    0    0    1    0    0    0
4  PID_233835  DERM_263067    0    0    1    0    0    0
